In [1]:
import os
import re
import pandas as pd
from itertools import combinations

# =========================
# Paths
# =========================
INPUT_DIR = r"../data/mapped-gene-second"
OUTPUT_DIR = r"../data/gene_combo"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# 1. Special-file detection (copy as-is)
# =========================
def is_special(name):
    name = name.upper()

    # Environmental files (excluded entirely)
    if re.search(r'\d+C$', name): return True      # temperature
    if "PH" in name: return True                  # pH
    if "UV" in name: return True                  # UV

    # Compound treatments (kept but not combined)
    if "_" in name: return True

    return False


# =========================
# 2. Dose parsing (very robust)
# =========================
def parse_drug_dose(name):
    name = name.upper()

    # Extract the trailing number (handles %, SEC, no separator)
    matches = re.findall(r'(\d+\.?\d*)', name)

    if not matches:
        return name, None

    dose = float(matches[-1])

    # Strip the dose part
    drug = re.sub(r'[-_]?\d+\.?\d*%?', '', name)
    drug = drug.strip("-_")

    return drug, dose


# =========================
# 3. Pick the "middle dose"
# =========================
def pick_middle_dose(files):
    drug_map = {}

    for f in files:
        name = os.path.splitext(f)[0]

        if is_special(name):
            continue

        drug, dose = parse_drug_dose(name)

        if dose is None:
            continue

        drug_map.setdefault(drug, []).append((f, dose))

    selected = {}

    for drug, items in drug_map.items():
        items_sorted = sorted(items, key=lambda x: x[1])

        mid_idx = len(items_sorted) // 2
        selected[drug] = items_sorted[mid_idx][0]

    return selected


# =========================
# 4. Read gene files
# =========================
def load_gene_file(path):
    df = pd.read_csv(path, sep=None, engine="python")
    df.columns = df.columns.str.strip()

    gene_col = df.columns[0]
    val_col = df.columns[1]

    df[gene_col] = df[gene_col].astype(str).str.lower()
    df = df[df[gene_col].str.startswith("b")]

    df["score"] = pd.to_numeric(df[val_col], errors="coerce")
    df = df.dropna(subset=["score"])

    return dict(zip(df[gene_col], df["score"]))


# =========================
# 5. Core: gene-combination strategy (custom)
# =========================
def combine_gene(a, b):
    # Same direction
    if a > 0 and b > 0:
        return max(a, b)

    if a < 0 and b < 0:
        return min(a, b)

    # Conflict -> inhibition wins
    if a < 0 and b > 0:
        return a * 0.7

    if a > 0 and b < 0:
        return b * 0.7

    return (a + b) / 2


# =========================
# 6. Main flow
# =========================
all_files = [f for f in os.listdir(INPUT_DIR) if f.endswith(".csv")]

# Special files (copy directly)
special_files = [f for f in all_files if is_special(os.path.splitext(f)[0])]

# Normal drugs (selected dose)
selected = pick_middle_dose(all_files)

print("✅ 中剂量选择:")
for k, v in selected.items():
    print(k, "->", v)

# =========================
# 6.1 Save special files (no combination)
# =========================
for f in special_files:
    src = os.path.join(INPUT_DIR, f)
    dst = os.path.join(OUTPUT_DIR, f)

    df = pd.read_csv(src, sep=None, engine="python")
    df.to_csv(dst, index=False)

    print("🟡 保留特殊:", f)


# =========================
# 6.2 Pairwise combinations
# =========================
drug_files = list(selected.values())

for f1, f2 in combinations(drug_files, 2):

    name1 = os.path.splitext(f1)[0]
    name2 = os.path.splitext(f2)[0]

    print(f"\n🔗 组合: {name1} + {name2}")

    g1 = load_gene_file(os.path.join(INPUT_DIR, f1))
    g2 = load_gene_file(os.path.join(INPUT_DIR, f2))

    all_genes = set(g1) | set(g2)

    combined = {}

    for g in all_genes:
        v1 = g1.get(g)
        v2 = g2.get(g)

        if v1 is None:
            combined[g] = v2
        elif v2 is None:
            combined[g] = v1
        else:
            combined[g] = combine_gene(v1, v2)

    out_df = pd.DataFrame({
        "Gene": list(combined.keys()),
        f"{name1}_{name2}": list(combined.values())
    })

    out_name = f"{name1}_{name2}.csv"
    out_path = os.path.join(OUTPUT_DIR, out_name)

    out_df.to_csv(out_path, index=False)

    print("✅ 保存:", out_name)

print("\n🎉 全部完成（中剂量 + 合理组合版）")

✅ 中剂量选择:
A -> A22-5.0.csv
ACRIFLAVINE -> ACRIFLAVINE-10.csv
ACTINOMYCIND -> ACTINOMYCIND-10.0.csv
AMIKACIN -> AMIKACIN-0.1.csv
AMOXICILLIN -> AMOXICILLIN-1.0.csv
AMPICILLIN -> AMPICILLIN-4.0.csv
AZIDOTHYMIDINE -> AZIDOTHYMIDINE-1.0.csv
AZITHROMYCIN -> AZITHROMYCIN-0.1.csv
AZTREONAM -> AZTREONAM-0.04.csv
BACITRACIN -> BACITRACIN-200.csv
BENZALKONIUM -> BENZALKONIUM-10.csv
BICYCLOMYCIN -> BICYCLOMYCIN-10.csv
BILE -> BILE-1.0%.csv
BLEOMYCIN -> BLEOMYCIN-1.0.csv
CARBENICILLIN -> CARBENICILLIN-1.0.csv
CCCP -> CCCP-0.5.csv
CECROPINB -> CECROPINB-0.3.csv
CEFACLOR -> CEFACLOR-2.0.csv
CEFOXITIN -> CEFOXITIN-0.75.csv
CEFSULODIN -> CEFSULODIN-18.0.csv
CEFTAZIDIME -> CEFTAZIDIME-0.05.csv
CERULENIN -> CERULENIN-4.0.csv
CHIR -> CHIR090-0.04.csv
CHLOROPROMAZINE -> CHLOROPROMAZINE-12.csv
CHOLATE -> CHOLATE-1.0%.csv
CIPROFLOXACIN -> CIPROFLOXACIN-0.006.csv
CISPLATIN -> CISPLATIN-50.csv
CLARYTHROMYCIN -> CLARYTHROMYCIN-5.0.csv
CYCLOSERINED -> CYCLOSERINED-16.csv
DEOXYCHOLATE -> DEOXYCHOLATE-0.5%.csv
DIB

In [2]:
import os
import re
import pandas as pd
from itertools import combinations

# =========================
# Paths
# =========================
INPUT_DIR = r"../data/mapped-gene-second"
OUTPUT_DIR = r"../data/gene_combo-2"
SPECIAL_DIR = os.path.join(OUTPUT_DIR, "special_conditions")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SPECIAL_DIR, exist_ok=True)

# =========================
# 1. Classification function (upgraded core)
# =========================
def classify_file(fname):
    name = os.path.splitext(fname)[0].upper().strip()

    # Temperature (excluded)
    if re.match(r"^\d+(\.\d+)?C$", name):
        return "ignore"

    # ❌ pH
    if re.match(r"^PH\d+", name):
        return "ignore"

    # ❌ UV
    if "UV" in name:
        return "ignore"

    # Multi-drug combination (contains _)
    if "_" in name:
        return "special"

    return "normal"


# =========================
# 2. Parse drug + dose
# =========================
def parse_drug_and_dose(fname):
    name = os.path.splitext(fname)[0].upper().strip()
    name = name.replace(" ", "")

    match = re.match(r"^(.+)-([\d\.]+)%?$", name)

    if match:
        drug = match.group(1)
        dose = float(match.group(2))
    else:
        drug = name
        dose = 0

    return drug, dose


# =========================
# 3. Grouping
# =========================
drug_dict = {}
special_files = []

for f in os.listdir(INPUT_DIR):
    if not f.endswith(".csv"):
        continue

    category = classify_file(f)

    if category == "ignore":
        continue

    elif category == "special":
        special_files.append(f)

    else:
        drug, dose = parse_drug_and_dose(f)
        drug_dict.setdefault(drug, []).append((f, dose))


# =========================
# 4. Middle-dose selection
# =========================
def select_middle(lst):
    lst = sorted(lst, key=lambda x: x[1])
    n = len(lst)

    if n == 1:
        return lst[0]
    elif n == 2:
        return lst[0]   # take the low dose
    else:
        return lst[n // 2]


drug_selected = {}

print("\n✅ 中剂量选择结果:")
for drug, lst in drug_dict.items():
    sel = select_middle(lst)
    drug_selected[drug] = sel
    print(f"{drug} -> {sel}")


# =========================
# 5. Read genes
# =========================
def load_gene_file(fname):
    df = pd.read_csv(os.path.join(INPUT_DIR, fname), sep=None, engine="python")

    df.columns = df.columns.str.strip()

    gene_col = df.columns[0]
    val_col = df.columns[1]

    df[gene_col] = df[gene_col].astype(str).str.strip().str.lower()
    df = df[df[gene_col].str.startswith("b")]

    df["fitness"] = pd.to_numeric(df[val_col], errors="coerce")
    df = df.dropna(subset=["fitness"])

    return dict(zip(df[gene_col], df["fitness"]))


# =========================
# 6. Gene combination (inhibition first)
# =========================
def combine_genes(g1, g2):
    all_genes = set(g1) | set(g2)
    combined = {}

    for g in all_genes:
        v1 = g1.get(g)
        v2 = g2.get(g)

        if v1 is None:
            combined[g] = v2
        elif v2 is None:
            combined[g] = v1
        else:
            if v1 * v2 > 0:
                combined[g] = (v1 + v2) / 2
            else:
                combined[g] = min(v1, v2)  # inhibition first

    return combined


# =========================
# 7. Pairwise combinations
# =========================
drug_list = list(drug_selected.keys())

print("\n🚀 开始组合...")

for d1, d2 in combinations(drug_list, 2):

    f1 = drug_selected[d1][0]
    f2 = drug_selected[d2][0]

    g1 = load_gene_file(f1)
    g2 = load_gene_file(f2)

    combined = combine_genes(g1, g2)

    df_out = pd.DataFrame({
        "Gene": list(combined.keys()),
        "fitness": list(combined.values())
    })

    name = f"{d1}__{d2}".upper()
    out_path = os.path.join(OUTPUT_DIR, f"{name}.csv")

    df_out.to_csv(out_path, index=False)

    print("✅", name)


# =========================
# 8. Save special cases separately
# =========================
print("\n📦 处理特殊条件（不参与组合）...")

for f in special_files:
    df = pd.read_csv(os.path.join(INPUT_DIR, f), sep=None, engine="python")

    df.columns = df.columns.str.strip()

    gene_col = df.columns[0]
    val_col = df.columns[1]

    df[gene_col] = df[gene_col].astype(str).str.strip().str.lower()
    df = df[df[gene_col].str.startswith("b")]

    df["fitness"] = pd.to_numeric(df[val_col], errors="coerce")
    df = df.dropna(subset=["fitness"])

    name = os.path.splitext(f)[0].upper()
    out_path = os.path.join(SPECIAL_DIR, f"{name}.csv")

    df[[gene_col, "fitness"]].to_csv(out_path, index=False)

    print("✅ 特殊:", name)


print("\n🎉 全部完成！")


✅ 中剂量选择结果:
A22 -> ('A22-5.0.csv', 5.0)
ACETATE -> ('ACETATE.csv', 0)
ACRIFLAVINE -> ('ACRIFLAVINE-2.csv', 2.0)
ACTINOMYCIND -> ('ACTINOMYCIND-10.0.csv', 10.0)
AMIKACIN -> ('AMIKACIN-0.1.csv', 0.1)
AMOXICILLIN -> ('AMOXICILLIN-1.0.csv', 1.0)
AMPICILLIN -> ('AMPICILLIN-4.0.csv', 4.0)
ANAEROBIC -> ('ANAEROBIC.csv', 0)
AZIDOTHYMIDINE -> ('AZIDOTHYMIDINE-1.0.csv', 1.0)
AZITHROMYCIN -> ('AZITHROMYCIN-0.1.csv', 0.1)
AZTREONAM -> ('AZTREONAM-0.02.csv', 0.02)
BACITRACIN -> ('BACITRACIN-200.csv', 200.0)
BENZALKONIUM -> ('BENZALKONIUM-10.csv', 10.0)
BICYCLOMYCIN -> ('BICYCLOMYCIN-1.csv', 1.0)
BILE -> ('BILE-1.0%.csv', 1.0)
BLEOMYCIN -> ('BLEOMYCIN-1.0.csv', 1.0)
CALCOFLUOR -> ('CALCOFLUOR.csv', 0)
CARBENICILLIN -> ('CARBENICILLIN-1.0.csv', 1.0)
CCCP -> ('CCCP-0.5.csv', 0.5)
CECROPINB -> ('CECROPINB-0.1.csv', 0.1)
CEFACLOR -> ('CEFACLOR-2.0.csv', 2.0)
CEFOXITIN -> ('CEFOXITIN-0.75.csv', 0.75)
CEFSULODIN -> ('CEFSULODIN-18.0.csv', 18.0)
CEFTAZIDIME -> ('CEFTAZIDIME-0.05.csv', 0.05)
CERULENIN -> ('

In [1]:
import os
import re
import pandas as pd
from itertools import combinations

# =========================
# Paths
# =========================
INPUT_DIR = r"../data/mapped-gene-second"
OUTPUT_DIR = r"../data/gene_combo-3"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# =========================
# 1. Special files
# =========================
def is_special(name):
    name = name.upper()

    if re.search(r'\d+C$', name): return True
    if "PH" in name: return True
    if "UV" in name: return True
    if "_" in name: return True

    return False


# =========================
# 2. Detect "pre-combined files"
# =========================
def is_precombined(name):
    return "," in name


# =========================
# 3. Clean the drug name (core fix)
# =========================
def clean_drug(name):
    name = name.upper()

    # Remove dose numbers
    name = re.sub(r'\d+\.?\d*%?', '', name)

    # Remove leftover symbols
    name = name.replace(",", "-")
    name = re.sub(r'[-_]+', '-', name)

    return name.strip("-_")


# =========================
# 4. Parse dose (for filtering only)
# =========================
def parse_dose(name):
    matches = re.findall(r'\d+\.?\d*', name)
    return float(matches[-1]) if matches else None


# =========================
# 5. Middle-dose selection (fixed)
# =========================
def pick_middle_dose(files):
    drug_map = {}

    for f in files:
        name = os.path.splitext(f)[0]

        if is_special(name):
            continue

        if is_precombined(name):
            continue  # key fix: skip pre-combined files

        drug = clean_drug(name)
        dose = parse_dose(name)

        if dose is None:
            continue

        drug_map.setdefault(drug, []).append((f, dose))

    selected = {}

    for drug, items in drug_map.items():
        items_sorted = sorted(items, key=lambda x: x[1])
        mid_idx = len(items_sorted) // 2
        selected[drug] = items_sorted[mid_idx][0]

    return selected


# =========================
# 6. Read genes
# =========================
def load_gene_file(path):
    df = pd.read_csv(path, sep=None, engine="python")
    df.columns = df.columns.str.strip()

    gene_col = df.columns[0]
    val_col = df.columns[1]

    df[gene_col] = df[gene_col].astype(str).str.lower()
    df = df[df[gene_col].str.startswith("b")]

    df["score"] = pd.to_numeric(df[val_col], errors="coerce")
    df = df.dropna(subset=["score"])

    return dict(zip(df[gene_col], df["score"]))


# =========================
# 7. Combination rules
# =========================
def combine_gene(a, b):
    if a > 0 and b > 0:
        return max(a, b)
    if a < 0 and b < 0:
        return min(a, b)
    if a < 0 and b > 0:
        return a * 0.7
    if a > 0 and b < 0:
        return b * 0.7
    return (a + b) / 2


# =========================
# 8. Main flow
# =========================
all_files = [f for f in os.listdir(INPUT_DIR) if f.endswith(".csv")]

# Special files
special_files = [f for f in all_files if is_special(os.path.splitext(f)[0])]

# Normal single drugs
selected = pick_middle_dose(all_files)

print("✅ 中剂量选择:")
for k, v in selected.items():
    print(k, "->", v)


# =========================
# 9. Save special files
# =========================
for f in special_files:
    src = os.path.join(INPUT_DIR, f)
    dst = os.path.join(OUTPUT_DIR, f)

    pd.read_csv(src).to_csv(dst, index=False)
    print("🟡 保留特殊:", f)


# =========================
# 10. Pairwise combinations (fixed)
# =========================
drug_files = list(selected.values())

for f1, f2 in combinations(drug_files, 2):

    name1 = clean_drug(os.path.splitext(f1)[0])
    name2 = clean_drug(os.path.splitext(f2)[0])

    combo_name = f"{name1}-{name2}"

    print(f"\n🔗 组合: {combo_name}")

    g1 = load_gene_file(os.path.join(INPUT_DIR, f1))
    g2 = load_gene_file(os.path.join(INPUT_DIR, f2))

    all_genes = set(g1) | set(g2)

    combined = {}

    for g in all_genes:
        v1 = g1.get(g)
        v2 = g2.get(g)

        if v1 is None:
            combined[g] = v2
        elif v2 is None:
            combined[g] = v1
        else:
            combined[g] = combine_gene(v1, v2)

    out_df = pd.DataFrame({
        "Gene": list(combined.keys()),
        combo_name: list(combined.values())
    })

    out_path = os.path.join(OUTPUT_DIR, combo_name + ".csv")
    out_df.to_csv(out_path, index=False)

    print("✅ 保存:", combo_name)

✅ 中剂量选择:
A -> A22-5.0.csv
ACRIFLAVINE -> ACRIFLAVINE-10.csv
ACTINOMYCIND -> ACTINOMYCIND-10.0.csv
AMIKACIN -> AMIKACIN-0.1.csv
AMOXICILLIN -> AMOXICILLIN-1.0.csv
AMPICILLIN -> AMPICILLIN-4.0.csv
AZIDOTHYMIDINE -> AZIDOTHYMIDINE-1.0.csv
AZITHROMYCIN -> AZITHROMYCIN-0.1.csv
AZTREONAM -> AZTREONAM-0.04.csv
BACITRACIN -> BACITRACIN-200.csv
BENZALKONIUM -> BENZALKONIUM-10.csv
BICYCLOMYCIN -> BICYCLOMYCIN-10.csv
BILE -> BILE-1.0%.csv
BLEOMYCIN -> BLEOMYCIN-1.0.csv
CARBENICILLIN -> CARBENICILLIN-1.0.csv
CCCP -> CCCP-0.5.csv
CECROPINB -> CECROPINB-0.3.csv
CEFACLOR -> CEFACLOR-2.0.csv
CEFOXITIN -> CEFOXITIN-0.75.csv
CEFSULODIN -> CEFSULODIN-18.0.csv
CEFTAZIDIME -> CEFTAZIDIME-0.05.csv
CERULENIN -> CERULENIN-4.0.csv
CHIR -> CHIR090-0.04.csv
CHLOROPROMAZINE -> CHLOROPROMAZINE-12.csv
CHOLATE -> CHOLATE-1.0%.csv
CIPROFLOXACIN -> CIPROFLOXACIN-0.006.csv
CISPLATIN -> CISPLATIN-50.csv
CLARYTHROMYCIN -> CLARYTHROMYCIN-5.0.csv
CYCLOSERINED -> CYCLOSERINED-16.csv
DEOXYCHOLATE -> DEOXYCHOLATE-0.5%.csv
DIB